# Estrategias de Escalado

Escalado horizontal, balanceo de carga y caché

## Introducción

Escalar sin datos es como medicar sin diagnóstico. Las decisiones de escalamiento deben basarse en métricas reales de rendimiento: carga del servidor, latencia de respuestas, y saturación de recursos. No se escala por corazonada ni por horario.

### Objetivos de Aprendizaje

- Entender qué señal debe guiar las decisiones de escalamiento
- Diferenciar entre escalado vertical y horizontal
- Configurar balanceador de carga con Nginx
- Implementar estrategias de caché para reducir carga
- Monitorear métricas clave para tomar decisiones de escalado

## Qué señal debe guiar una decisión de escalamiento

> Las decisiones de escalamiento deben guiarse por métricas reales: carga de CPU, latencia de respuesta, uso de memoria, y saturación de la conexión a la base de datos. Nunca por intuición o por el reloj.

In [ ]:
print("=== Señales de Escalamiento ===")

metricas_clave = {
    "CPU Usage": "Porcentaje de uso de CPU > 80% sustained",
    "Latency": "Tiempo de respuesta > 500ms p95",
    "Memory": "RAM utilizada > 85%",
    "DB Connections": "Conexiones cercanas al límite del pool",
    "Request Queue": "Requests en cola > 100",
    "Error Rate": "Tasa de errores 5xx > 1%",
}

for metrica, desc in metricas_clave.items():
    print(f"  {metrica}: {desc}")

print("""
ESCALAR POR META (umbral superado) NO POR HORARIO
Incorrecto: "Escalamos a las 3pm porque es cuando hay tráfico"
Correcto:   "Escalamos cuando CPU > 80% por 5+ minutos"

Las métricas eliminan la subjetividad y permiten
decisiones automáticas y reproducibles.
""")

## Escalado Vertical vs Horizontal

> Escalado vertical: agregar más recursos (CPU, RAM) al servidor existente. Escalado horizontal: agregar más servidores para distribuir la carga.

In [ ]:
print("=== Escalado Vertical vs Horizontal ===")

comparacion = {
    "Vertical": {
        "Definicion": "Más CPU/RAM en el mismo servidor",
        "Pros": "Simple, sin cambios en código",
        "Contras": "Límite físico, punto único de fallo",
        "Cuando": "Cuellos de botella en recursos simples",
    },
    "Horizontal": {
        "Definicion": "Más servidores idénticos detrás de balanceador",
        "Pros": "Escala ilimitada, redundancia",
        "Contras": "Requiere stateless app, sesión compartida",
        "Cuando": "Alto tráfico, alta disponibilidad",
    },
}

for tipo, info in comparacion.items():
    print(f"\n{tipo}:")
    for k, v in info.items():
        print(f"  {k}: {v}")

print("""
Ejemplo vertical:
  2 vCPU → 8 vCPU (upgrade de instancia)

Ejemplo horizontal:
  1 servidor → 4 servidores detrás de Nginx
""")

## Balanceador de Carga con Nginx

> Nginx puede actuar como balanceador de carga, distribuyendo requests entre múltiples servidores upstream. Soporta algoritmos: round-robin, least_conn, ip_hash.

In [ ]:
nginx_config = """
# /etc/nginx/conf.d/load-balancer.conf

upstream backend {
    least_conn;  # Envía al servidor con menos conexiones
    
    server 10.0.0.2:8000;
    server 10.0.0.3:8000;
    server 10.0.0.4:8000;
    
    keepalive 32;  # Mantiene conexiones abiertas
}

server {
    listen 80;
    server_name miapp.com;

    location / {
        proxy_pass http://backend;
        
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
        
        # Timeouts para evitar que conexiones lentas saturen
        proxy_connect_timeout 5s;
        proxy_read_timeout 30s;
    }
}
"""

print("=== Configuración Nginx como Balanceador ===")
print(nginx_config)

algoritmos = {
    "round-robin": "Distribución equitativa (default)",
    "least_conn": "Al servidor con menos conexiones activas",
    "ip_hash": "Por IP del cliente (para sesiones sticky)",
    "weighted": "Según peso definido (server weight=2)",
}

print("Algoritmos de balanceo:")
for alg, desc in algoritmos.items():
    print(f"  {alg}: {desc}")

## Health Checks y Alta Disponibilidad

> El balanceador debe verificar que los servidores upstream están saludables antes de enviarles tráfico. Si un servidor falla, se retira del pool automáticamente.

In [ ]:
health_check_config = """
upstream backend {
    server 10.0.0.2:8000 max_fails=3 fail_timeout=30s;
    server 10.0.0.3:8000 max_fails=3 fail_timeout=30s;
    server 10.0.0.4:8000 max_fails=3 fail_timeout=30s;
}
"""

print("=== Health Checks ===")
print(health_check_config)

print("""
Parámetros de salud:
  max_fails=3: Sacar servidor tras 3 fallos consecutivos
  fail_timeout=30s: Tiempo antes de reintentar

Health check a nivel de aplicación (más robusto):
  location /health {
      proxy_pass http://backend;
      proxy_connect_timeout 2s;
      proxy_next_upstream error timeout;
  }
""")

print("Flujo de failover:")
pasos_failover = [
    "1. Servidor 10.0.0.3 deja de responder",
    "2. Nginx detecta 3 fallos consecutivos",
    "3. Servidor se marca como down (fail_timeout)",
    "4. Tráfico se redistribuye a .2 y .4",
    "5. Después de 30s, se reintenta una request",
    "6. Si recupera, vuelve al pool; si falla, se retira de nuevo",
]
for paso in pasos_failover:
    print(f"  {paso}")

## Estrategias de Caché

> La caché reduce la carga en base de datos y servidores al almacenar respuestas frecuentes. Tipos: CDN (estáticos), Nginx cache, Redis/Memcached (aplicación).

In [ ]:
print("=== Estrategias de Caché ===")

tipos_cache = {
    "Nginx FastCGI Cache": "Cachea respuestas de aplicaciones PHP/Python",
    "Redis/Memcached": "Cache en memoria para sesiones y queries",
    "CDN": "Contenido estático cerca del usuario (CloudFlare, S3+CloudFront)",
    "Browser Cache": "Headers Cache-Control y ETag locales",
}

for tipo, desc in tipos_cache.items():
    print(f"  {tipo}: {desc}\n")

nginx_cache = """
# Configuración Nginx con caché
proxy_cache_path /var/cache/nginx levels=1:2 keys_zone=app_cache:10m;

server {
    location / {
        proxy_pass http://backend;
        proxy_cache app_cache;
        proxy_cache_valid 200 5m;  # Cachear respuestas 200 por 5 min
        proxy_cache_valid 404 1m;
        add_header X-Cache-Status $upstream_cache_status;
    }
}
"""

print("Ejemplo Nginx FastCGI Cache:")
print(nginx_cache)

## Redis como Caché de Aplicación

> Redis almacena datos frecuentemente accedidos en memoria, reduciendo consultas a la base de datos. Ejemplo: cachear el resultado de queries pesadas.

In [ ]:
print("=== Redis como Caché ===")

redis_example = """
import redis
import json

r = redis.Redis(host='localhost', port=6379, db=0)

def get_user_with_cache(user_id):
    cache_key = f"user:{user_id}"
    
    # 1. Intentar obtener de caché
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached)
    
    # 2. Si no está en caché, consultar DB
    user = db.query("SELECT * FROM users WHERE id = %s", user_id)
    
    # 3. Guardar en caché por 5 minutos
    r.setex(cache_key, 300, json.dumps(user))
    
    return user

# Invalidez de caché cuando hay actualización
def update_user(user_id, data):
    db.execute("UPDATE users SET ... WHERE id = %s", user_id, data)
    r.delete(f"user:{user_id}")  # Invalidar caché
"""

print(redis_example)

ttl_examples = {
    "Sesiones de usuario": "15-30 minutos",
    "Queries frecuentes": "1-5 minutos",
    "Listas de productos": "5-15 minutos",
    "Dashboards analytics": "10-60 minutos",
}

print("TTL típicos por tipo de dato:")
for dato, ttl in ttl_examples.items():
    print(f"  {dato}: {ttl}")

## Auto-Scaling Basado en Métricas

> El auto-scaling ajusta automáticamente el número de instancias según métricas en tiempo real. Se define un mínimo, máximo, y regla de escalado.

In [ ]:
autoscaling_rules = """
# Ejemplo con AWS Auto Scaling Groups

Regla de escalado horizontal:
  Condición: CPU > 70% durante 5 minutos
  Acción: Agregar 1 instancia (máx 10)

Regla de escalado negativo:
  Condición: CPU < 30% durante 15 minutos
  Acción: Remover 1 instancia (mín 2)

# O con Docker + monitoring:
docker compose up -d --scale app=4

# Nginx upstream dinámico:
upstream backend {
    least_conn;
    server 10.0.0.2:8000;
    server 10.0.0.3:8000;
    # Más instancias se agregan dinámicamente
}
"""

print("=== Auto-Scaling ===")
print(autoscaling_rules)

metricas_autoscaling = {
    "CPU > 70%": "Escalar arriba",
    "CPU < 30%": "Escalar abajo",
    "Latencia p99 > 1s": "Escalar arriba",
    "Cola de requests > 100": "Escalar arriba",
}

print("Métricas para auto-scaling:")
for metrica, accion in metricas_autoscaling.items():
    print(f"  {metrica}: {accion}")

## Tips y Mejores Prácticas

> Escala según métricas, no según el reloj. CPU, latencia y saturación son las señales más confiables.

> Diseña para ser stateless: guarda sesiones en Redis, archivos en S3, no en disco local.

> Usa缓存 para reducir carga antes de escalar.缓存 es más barato que instancias nuevas.

> Define mínimo y máximo de instancias. Escalar a 0 es peligroso para aplicaciones con tráfico.

> El warm-up de instancias es costoso. Mejor tener 2 instancias siempre que 1 que escala a 3 bajo demanda.

> Monitoriza el balanceo: si una instancia recibe mucho más tráfico que otra, revisa el algoritmo.

## Errores Comunes

### Escalar sin métricas

¿Por qué ocurre?
- Decisión basada en intuición ("creo que va a crecer") o horario ("es hora pico").

Solución
- Define umbrales claros (CPU > 70%) y usa herramientas de monitoreo para detectarlos.

### Sesiones locales sin共享

¿Por qué ocurre?
- Usuario hace login en servidor A, siguiente request va a servidor B donde no existe sesión.

Solución
- Usar Redis para sesiones compartidas, o ip_hash en Nginx para sesiones sticky.

### Caché sin invalidación

¿Por qué ocurre?
- Se actualiza un producto pero el caché sigue mostrando el precio antiguo.

Solución
- Define TTL adecuados y mechanisms de invalidación cuando hay cambios.

### No probar el failover

¿Por qué ocurre?
- Se confía que si un servidor falla, el balanceador lo detectará, pero no se probó.

Solución
- Haz pruebas de chaos: mata una instancia y verifica que el tráfico se redistribuye.

### Sobre-escalar por spikes temporales

¿Por qué ocurre?
- Un spike de 1 minuto dispara escalado a 10 instancias, costing dinero innecesario.

Solución
- Usa métricas sustained (5+ minutos) y considera circuitos breakers para spikes.